# שבוע 11: סטטיסטיקה מתקדמת

בשיעור זה נלמד:
- MANOVA בפרמוטציה — תיאוריה מעמיקה ו-R²
- אלומטריה: קורלציה בין גודל לצורה
- השוואות מרובות ותיקון בונפרוני
- גרף חום (heatmap) של מטריצת p-values

> **מטרה**: החלו את הכלים הללו על מאגר הנתונים של הפרויקט הסופי שלכם.

In [ ]:
!pip install morphops python-bidi -q
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
rtl = get_display
print('הכל מוכן!')

## 1. MANOVA בפרמוטציה — תיאוריה

**למה לא t-test רגיל?**

נתוני צורה הם **רב-משתניים** — כל פרט הוא נקודה בעשרות ממדים. t-test פועל על משתנה אחד בלבד.

**MANOVA** = Multivariate ANOVA — בודק אם הממוצע הווקטורי שונה בין קבוצות.

**למה פרמוטציה ולא MANOVA קלאסי?**
- MANOVA מניח נורמליות רב-משתנית
- נתוני צורה לרוב אינם עומדים בהנחה זו
- פרמוטציה חופשייה מהנחות פרמטריות

**R² = effect size**: אחוז השונות המוסבר על ידי הקבוצה
- R² < 0.05 = אפקט קטן
- R² 0.05–0.15 = אפקט בינוני
- R² > 0.15 = אפקט גדול

In [ ]:
def permutation_manova(X, groups, n_perm=999, seed=42):
    """
    MANOVA בפרמוטציה.
    מחזיר: F (observed), p-value, R² (effect size)
    """
    np.random.seed(seed)

    def f_stat(X, g):
        unique_g = np.unique(g)
        gm = X.mean(axis=0)
        between = sum(
            np.sum(g == u) * np.sum((X[g == u].mean(0) - gm) ** 2)
            for u in unique_g
        )
        within = sum(
            np.sum((X[g == u] - X[g == u].mean(0)) ** 2)
            for u in unique_g
        )
        return between / within if within > 0 else 0

    obs = f_stat(X, groups)
    perm = [f_stat(X, np.random.permutation(groups)) for _ in range(n_perm)]
    p = (np.sum(np.array(perm) >= obs) + 1) / (n_perm + 1)
    r2 = obs / (obs + 1)
    return obs, p, r2, np.array(perm)

# דוגמה: שלוש קבוצות
np.random.seed(42)
X1 = np.random.randn(25, 10) + 0.5
X2 = np.random.randn(30, 10)
X3 = np.random.randn(20, 10) - 0.3
X_demo = np.vstack([X1, X2, X3])
g_demo = np.array(['קבוצה א'] * 25 + ['קבוצה ב'] * 30 + ['קבוצה ג'] * 20)

f_obs, p_val, r2, perm_dist = permutation_manova(X_demo, g_demo)
print(f'F = {f_obs:.3f}')
print(f'p = {p_val:.3f}')
print(f'R² = {r2:.3f} ({r2*100:.1f}% שונות מוסברת)')

In [ ]:
# ויזואליזציה של התפלגות הפרמוטציה
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(perm_dist, bins=40, color='lightblue', edgecolor='white', alpha=0.8,
        label=rtl('התפלגות H₀ (פרמוטציה)'))
ax.axvline(f_obs, color='red', linewidth=2.5,
           label=rtl(f'F נצפה = {f_obs:.2f}'))
ax.set_xlabel(rtl('סטטיסטיקת F'))
ax.set_ylabel(rtl('תדירות'))
ax.set_title(rtl(f'התפלגות פרמוטציה (p = {p_val:.3f})'))
ax.legend()
plt.tight_layout()
plt.show()

## 2. אלומטריה: גודל וצורה

**אלומטריה** = הקשר בין גודל לצורה.

- גדולים יותר → צורה שונה = **אלומטריה חיובית**
- אין קשר = **איזומטריה** (צורה עצמאית מגודל)

בודקים עם רגרסיה: PC1_צורה ~ ln(גודל-צנטרואיד)

In [ ]:
from scipy import stats

np.random.seed(99)
n = 65
centroid_size = np.random.uniform(50, 150, n)
pc1_shape = 0.004 * centroid_size + np.random.randn(n) * 0.12  # אלומטריה מתונה
pc2_shape = np.random.randn(n) * 0.12                            # ללא אלומטריה

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for ax, y, title_he in [
    (ax1, pc1_shape, 'PC1 — עם אלומטריה'),
    (ax2, pc2_shape, 'PC2 — ללא אלומטריה')
]:
    slope, intercept, r, p, se = stats.linregress(np.log(centroid_size), y)
    ax.scatter(np.log(centroid_size), y, alpha=0.6, color='steelblue', s=55)
    x_line = np.linspace(np.log(centroid_size).min(), np.log(centroid_size).max(), 100)
    ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2)
    ax.set_xlabel(rtl('ln(גודל-צנטרואיד)'))
    ax.set_ylabel(rtl('ציון PC (צורה)'))
    ax.set_title(rtl(f'{title_he}\nr={r:.3f}, p={p:.3f}'), fontsize=11)

plt.suptitle(rtl('בדיקת אלומטריה'), fontsize=13)
plt.tight_layout()
plt.show()

## 3. השוואות מרובות ותיקון בונפרוני

עם **k קבוצות** יש **k(k-1)/2 זוגות** לבדיקה.

**בעיית ריבוי הבדיקות**: אם בודקים 10 זוגות ב-α=0.05, מצפים למצוא ~0.5 תוצאות חיוביות שגויות.

**תיקון בונפרוני**: α' = α / מספר_הבדיקות

In [ ]:
from itertools import combinations

# נתוני דוגמה: 4 תקופות כרונולוגיות
np.random.seed(42)
periods = ['EBA', 'MBA', 'LBA', 'IA']
shifts = {'EBA': 0.0, 'MBA': 0.4, 'LBA': 0.7, 'IA': 0.2}
period_data = {p: np.random.randn(22, 6) + shifts[p] for p in periods}

n_comparisons = len(periods) * (len(periods) - 1) // 2
alpha_bonferroni = 0.05 / n_comparisons
print(f'מספר השוואות: {n_comparisons}')
print(f'α מתוקן (בונפרוני): {alpha_bonferroni:.4f}')
print()

results = []
for g1, g2 in combinations(periods, 2):
    X_comb = np.vstack([period_data[g1], period_data[g2]])
    g_comb = np.array([g1] * len(period_data[g1]) + [g2] * len(period_data[g2]))
    f_val, p_val, r2_val, _ = permutation_manova(X_comb, g_comb, n_perm=499)
    sig = '*' if p_val < alpha_bonferroni else ''
    results.append((g1, g2, p_val, r2_val, sig))
    print(f'  {g1} vs {g2}: p={p_val:.3f}, R²={r2_val:.3f} {sig}')

print(f'\nמובהק לאחר בונפרוני: {sum(r[4]=="*" for r in results)}/{n_comparisons}')

In [ ]:
# גרף חום של מטריצת p-values
n = len(periods)
p_matrix = np.ones((n, n))
for g1, g2, p, r2, sig in results:
    i, j = periods.index(g1), periods.index(g2)
    p_matrix[i, j] = p
    p_matrix[j, i] = p

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(p_matrix, cmap='RdYlGn_r', vmin=0, vmax=0.1)

for i in range(n):
    for j in range(n):
        if i != j:
            val = p_matrix[i, j]
            marker = ' *' if val < alpha_bonferroni else ''
            color = 'white' if val < 0.04 else 'black'
            ax.text(j, i, f'{val:.3f}{marker}', ha='center', va='center',
                   fontsize=10, color=color, fontweight='bold' if marker else 'normal')
        else:
            ax.text(j, i, '—', ha='center', va='center', fontsize=12, color='gray')

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(periods, fontsize=12)
ax.set_yticklabels(periods, fontsize=12)
ax.set_title(rtl(f'מטריצת p-values בין תקופות\n(* מובהק לאחר בונפרוני, α={alpha_bonferroni:.4f})'),
             fontsize=11)
plt.colorbar(im, label='p-value')
plt.tight_layout()
plt.show()

## יישום על מאגר הנתונים של הפרויקט הסופי

החליפו את `X_project` ו-`groups_project` בנתוני הפרויקט שלכם:

In [ ]:
# ─── הכניסו כאן את הנתונים שלכם ───
# X_project = ...     # מערך NumPy: (n_specimens, n_features)
# groups_project = ... # מערך NumPy של מחרוזות עם תוויות הקבוצות

# דוגמה (מחקו ושנו):
X_project = X_demo
groups_project = g_demo
# ─────────────────────────────────

f_proj, p_proj, r2_proj, _ = permutation_manova(X_project, groups_project)
print('תוצאות הפרויקט:')
print(f'  F = {f_proj:.3f}')
print(f'  p = {p_proj:.3f}')
print(f'  R² = {r2_proj:.3f} ({r2_proj*100:.1f}%)')

## תרגיל

1. למה פרמוטציה עדיפה על MANOVA קלאסי בניתוח נתוני צורה?
2. בניתוח שלכם עם 3 קבוצות, כמה השוואות זוגיות תצטרכו? מה יהיה α המתוקן?
3. אם מצאתם אלומטריה, מה המשמעות הארכאולוגית?
4. מה ה-R² המינימלי שאתם מחשיבים כ'אפקט משמעותי' בארכאולוגיה? הנמקו.